In [ ]:
import os

# change working directory so that it picks up the grapehelper library
os.chdir("/home/ftorgano/rna-kg-analysis")
print(os.getcwd())

In [ ]:
import matplotlib.pyplot as plt
cycler_colors = ["#3f90da", "#ffa90e", "#bd1f01", "#94a4a2", "#832db6", "#a96b59", "#e76300", "#b9ac70", "#717581", "#92dadd"]
from cycler import cycler
plt.rcParams['axes.prop_cycle'] = cycler(color=cycler_colors)

# Node & Edge type count tables

In [ ]:
import pandas as pd

path_to_views = '/home/ftorgano/rna-kg-analysis/RNA-KG/Views/view'
def load_view(view_name):
    df_nodes = pd.read_pickle(path_to_views + view_name + '/nodes.pkl')
    df_edges = pd.read_pickle(path_to_views + view_name + '/edges.pkl')
    # add node type column to edges
    df_edges = df_edges.join(df_nodes.set_index('name'), on='subject', rsuffix='_subject').join(df_nodes.set_index('name'), on='object', rsuffix='_object')
    df_edges['types'] = df_edges['type_subject'] + ' - ' + df_edges['type_object']
    return {
        'nodes': df_nodes,
        'edges': df_edges
    }

In [ ]:
# load pkl using pandas
view0 = load_view('0')

In [ ]:
view0['nodes'].head()

# Node & Edge type count tables

In [ ]:
# generate a function that counts the number of node types and outputs a latex table
def generate_node_types_table(df_nodes, view_number):
    out_string = f"""\\begin{{table}}[!htbp]
\\caption{{Node type counts for view{view_number}}}
\\centering
\\begin{{tabular}}{{|l|r|}}
\\hline
\\bf{{Node type}} & \\bf{{Count}}  \\\\
\\hline
"""
    counts = df_nodes['type'].value_counts()
    for i in range(len(counts)):
        out_string += f"{counts.index[i]} & {counts.iloc[i]} \\\\ \n"
    out_string += f"""        \\hline
    \\end{{tabular}}
    \\label{{rna-kg:tab:node-comp:view{view_number}}}
\\end{{table}}
"""
    return out_string
# generate a function that counts the number of edge types and outputs a latex table
def generate_edge_types_table(df_edges, view_number):
    out_string = f"""\\begin{{table}}[!htbp]
\\caption{{Edge source-target type counts for view{view_number}}}
\\centering
\\begin{{tabular}}{{|l|l|r|l|}}
\\hline
\\bf{{Source type}} & \\bf{{Target type}} & \\bf{{Count}} & \\bf{{Edge types}}  \\\\
\\hline
"""
    counts = df_edges['types'].value_counts()
    for i in range(len(counts)):
        edge_types = df_edges[df_edges['types'] == counts.index[i]]['type'].unique()
        edge_types_str = '\\begin{tabular}[c]{@{}l@{}}' +'\\\\ '.join(edge_types) + '\\end{tabular}'
        
        # don't print \hline on the last row
        if i == len(counts) - 1:
            out_string += f'{str(counts.index[i]).replace("-","&")} & {counts.iloc[i]} & {edge_types_str}\\\\ \n'
        else:
            out_string += f'{str(counts.index[i]).replace("-","&")} & {counts.iloc[i]} & {edge_types_str}\\\\ \\hline \n'
    out_string+=f"""        \\hline
    \\end{{tabular}}
    \\label{{rna-kg:tab:edge-comp:view{view_number}}}
\\end{{table}}
"""
    return out_string

In [ ]:
print(generate_node_types_table(view0['nodes'],0))

In [ ]:
print(generate_edge_types_table(view0['edges'],0))

In [ ]:
with open('RNA-KG_notebooks/Views/views_tables2.tex','w+') as f:
    with open('RNA-KG_notebooks/Views/views_tables2_only_edges.tex','w+') as f2:
        for view_number in [0,4,9,12,14,16,17]:
            print(view_number)
            view = load_view(str(view_number))
            f.write(f'% View{view_number}\n')
            f.write(generate_node_types_table(view['nodes'],view_number))
            f.write('\n')
            edges_table = generate_edge_types_table(view['edges'],view_number)
            f.write(edges_table)
            f2.write(edges_table)
            f.write('\n\n')
            f2.write('\n\n')

# Node degree histogram

In [ ]:
# function that generates a node degree histogram
def calculate_node_degree(df_edges, indegree = False, outdegree = True):
    if indegree == False and outdegree == False:
        print('Please specify indegree or outdegree or both')
        return
    if indegree:
        degrees = df_edges['object'].value_counts()
    elif outdegree:
        degrees = df_edges['subject'].value_counts()
    if indegree and outdegree:
        degrees = degrees.add(df_edges['subject'].value_counts(), fill_value=0)
    return degrees.value_counts().sort_index()

def generate_node_degree_histogram(df_degrees, logscale = True, save_location = None, title=None, show=True):
    fig = plt.figure()
    fig.set_tight_layout(True)
    plt.bar(df_degrees.index, df_degrees)
    ax = fig.get_axes()
    if logscale:
        ax[0].set_yscale('log')
    plt.xlabel('Degree')
    plt.ylabel('Number of nodes')
    if title:
        plt.title(title)
    # check if save_location is a list
    if isinstance(save_location, list):
        for loc in save_location:
            plt.savefig(loc, dpi=600)
    elif save_location:
        plt.savefig(save_location, dpi=600)
    if show:
        plt.show()

In [ ]:
df_degrees = calculate_node_degree(view0['edges'], indegree = True, outdegree = True)
generate_node_degree_histogram(df_degrees, logscale = True, save_location = 'RNA-KG_notebooks/Views/NodeDegreePlots/degDist0Test.png')

In [ ]:
map_number_to_title = {
    0: 'miRNAdisease', # miRNAdisease\textsubscript{view}
    4: 'miRNAdiseaseSO',  # miRNAdiseaseSO\textsubscript{view}
    9: 'GOdisease', # GOdisease\textsubscript{view}
    11: 'GOdiseaseExtended',
    12: 'piRNAdisease', # piRNAdisease\textsubscript{view}
    14: 'piRNAdiseaseGO', # piRNAdiseaseGO\textsubscript{view}
    16: 'chemicalDisease', # chemicalDisease\textsubscript{view}
    17: 'cellAnatomy' # cellAnatomy\textsubscript{view}
}

In [ ]:
for view_number in [0,4,9,12,14,16,17]:
    print(view_number)
    view = load_view(str(view_number))
    df_degrees = calculate_node_degree(view['edges'], indegree = True, outdegree = True)
    generate_node_degree_histogram(df_degrees, logscale = True, 
                                   save_location = [f'RNA-KG_notebooks/Views/NodeDegreePlots/degDist{view_number}.pdf',
                                                    f'RNA-KG_notebooks/Views/NodeDegreePlots/degDist{view_number}.png'],
                                   show=False)

# Results bar plot

In [ ]:
views = {
    0: {'name': 'miRNAdisease', 'title': 'miRNAdisease$_{v}$', 'type_pairs':[('miRNA','Disease'), ('miRNA', 'Phenotype')]},
    4: {'name': 'miRNAdiseaseSO', 'title': 'miRNAdiseaseSO$_{v}$', 'type_pairs':[('miRNA','Disease'), ('miRNA', 'Phenotype')]},
    9: {'name': 'GOdisease', 'title': 'GOdisease$_{v}$', 'type_pairs':[('lncRNA', 'GO'), ('miRNA','GO')]},
    12: {'name': 'piRNAdisease', 'title': 'piRNAdisease$_{v}$', 'type_pairs':[('piRNA','Disease')]},
    14: {'name': 'piRNAdiseaseGO', 'title': 'piRNAdiseaseGO$_{v}$', 'type_pairs':[('piRNA','Disease')]},
    16: {'name': 'chemicalDisease', 'title': 'chemicalDisease$_{v}$', 'type_pairs':[('Chemical','Disease')]},
    17: {'name': 'cellAnatomy', 'title': 'cellAnatomy$_{v}$', 'type_pairs':[('lncRNA','Anatomy'),('lncRNA','Cell')]}
}
views.keys()

## LINE

In [ ]:
means_tree = []
means_forest = []
stds_tree = []
stds_forest = []
labels = []
for (key, value) in views.items():
    results_view = pd.read_csv(f'RNA-KG_notebooks/Views/view{key}/csv/result_custom_filtered_final.csv', index_col=0)
    results_view_tree = results_view[results_view['Model']=='Decision Tree Classifier']
    results_view_forest = results_view[results_view['Model']=='Random Forest Classifier']
    for type_pair in value['type_pairs'][:1]:
        results_view_tree_specific = results_view_tree[
            (results_view_tree['Source Type']==type_pair[0]) 
            & (results_view_tree['Destination Type']==type_pair[1])] 
        results_view_forest_specific = results_view_forest[
            (results_view_forest['Source Type']==type_pair[0]) 
            & (results_view_forest['Destination Type']==type_pair[1])] 
        means_tree.append(results_view_tree_specific['Mean balanced accuracy'].mean())
        stds_tree.append(results_view_tree_specific['Mean balanced accuracy'].std())
        means_forest.append(results_view_forest_specific['Mean balanced accuracy'].mean())
        stds_forest.append(results_view_forest_specific['Mean balanced accuracy'].std())
        labels.append(f"{type_pair[0]}-{type_pair[1]}\n{value['title']}")

In [ ]:
plt.figure(figsize=(10,5))
fig, ax = plt.subplots(layout='constrained')
rects = plt.bar(labels, means_forest, yerr=stds_forest, width=0.5)
ax.bar_label(rects, padding=3, fmt='{:.1%}')
plt.ylabel('Overall Balanced Accuracy')
plt.xticks(rotation=45)
plt.ylim(0,1.03)
plt.savefig('RNA-KG_notebooks/Views/ResultsPlots/OverallBalancedAccuracy2.pdf')
plt.show()

In [ ]:
plt.figure(figsize=(10,5))
fig, ax = plt.subplots(layout='constrained')

x = list(range(len(labels)))  # the label locations
width = 0.45  # the width of the bars
multiplier = 0

rects = plt.bar(x, means_tree, yerr=stds_tree, width=width, label='Decision Tree')
ax.bar_label(rects, padding=3, fmt='{:.1%}', rotation='vertical',label_type='center')
rects = plt.bar([v+width for v in x], means_forest, yerr=stds_forest, width=width, label='Random Forest')
ax.bar_label(rects, padding=3, fmt='{:.1%}', rotation='vertical',label_type='center')

plt.ylabel('Overall Balanced Accuracy')
plt.xticks(rotation=45)
ax.set_xticks([v+width/2 for v in x], labels)
plt.legend(loc='upper center', bbox_to_anchor=(0.5, 1.2), ncol=3, title='Prediction Model')
plt.ylim(0,1.03)

plt.savefig('RNA-KG_notebooks/Views/ResultsPlots/LinkPrediction_OverallBalancedAccuracy_LINE_TreeForest.pdf')
plt.show()

## Node2Vec BFS

In [ ]:
means_tree = []
means_forest = []
stds_tree = []
stds_forest = []
labels = []
for (key, value) in views.items():
    results_view = pd.read_csv(f'RNA-KG_notebooks/Views/view{key}/csv/result_custom_filtered_final_node2vecBFS.csv', index_col=0)
    results_view_tree = results_view[results_view['Model']=='Decision Tree Classifier']
    results_view_forest = results_view[results_view['Model']=='Random Forest Classifier']
    for type_pair in value['type_pairs'][:1]:
        results_view_tree_specific = results_view_tree[
            (results_view_tree['Source Type']==type_pair[0]) 
            & (results_view_tree['Destination Type']==type_pair[1])] 
        results_view_forest_specific = results_view_forest[
            (results_view_forest['Source Type']==type_pair[0]) 
            & (results_view_forest['Destination Type']==type_pair[1])] 
        means_tree.append(results_view_tree_specific['Mean balanced accuracy'].mean())
        stds_tree.append(results_view_tree_specific['Mean balanced accuracy'].std())
        means_forest.append(results_view_forest_specific['Mean balanced accuracy'].mean())
        stds_forest.append(results_view_forest_specific['Mean balanced accuracy'].std())
        labels.append(f"{type_pair[0]}-{type_pair[1]}\n{value['title']}")

In [ ]:
plt.figure(figsize=(10,5))
fig, ax = plt.subplots(layout='constrained')
rects = plt.bar(labels, means_forest, yerr=stds_forest, width=0.5)
ax.bar_label(rects, padding=3, fmt='{:.1%}')
plt.ylabel('Overall Balanced Accuracy')
plt.xticks(rotation=45)
plt.ylim(0,1.03)
plt.savefig('RNA-KG_notebooks/Views/ResultsPlots/OverallBalancedAccuracy2_node2vec.pdf')
plt.show()

In [ ]:
plt.figure(figsize=(10,5))
fig, ax = plt.subplots(layout='constrained')

x = list(range(len(labels)))  # the label locations
width = 0.4  # the width of the bars
multiplier = 0

rects = plt.barh(x, means_tree, yerr=stds_tree, label='Decision Tree', height=width)
ax.bar_label(rects, padding=3, fmt='{:.1%}', label_type='center')
rects = plt.barh([v+width for v in x], means_forest, yerr=stds_forest, label='Random Forest',height=width)
ax.bar_label(rects, padding=3, fmt='{:.1%}', label_type='center')

plt.xlabel('Overall Balanced Accuracy')
#plt.xticks(rotation=45)
ax.set_yticks([v+width/2 for v in x], labels)
plt.legend(loc='upper center', bbox_to_anchor=(0.5, -0.1), ncol=3, title='Edge Prediction Model')
plt.xlim(0,1.03)

plt.savefig('RNA-KG_notebooks/Views/ResultsPlots/LinkPrediction_OverallBalancedAccuracy_N2V_TreeForest.pdf')
plt.savefig('RNA-KG_notebooks/Views/ResultsPlots/LinkPrediction_OverallBalancedAccuracy_N2V_TreeForest.png', dpi=450)
plt.show()